# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides an example for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install --quiet mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset
dataset = mlc.Dataset(url)
# Access metadata as an object
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, columns and their `@id`s.

The Croissant schema enables referencing all entities (record sets, fields, columns) by their `@id`. Let's list all available record sets with their `@id` and field information.

In [ ]:
# List all record sets and their fields by @id
record_sets = list(metadata.record_sets)
print(f"Found {len(record_sets)} record set(s):\n")
for record_set in record_sets:
    print(f"- RecordSet name: {record_set.name}")
    print(f"  @id: {record_set.id}")
    print(f"  Description: {record_set.description}")
    print("  Fields:")
    for field in record_set.fields:
        print(f"    - {field.name:30} @id: {field.id}")
    print("")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from above overview.

We'll select the main record set, which contains the mass spectrometry analysis results.

In [ ]:
# Extract data from the main record set
# For this FAIR^2 dataset, the record set id is typically 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json#record-set-1'
# Let's enumerate record set ids
main_record_set_id = record_sets[0].id
print(f"Using RecordSet @id: {main_record_set_id}\n")

# Extract data for each record set
dataframes = {}
for rs in record_sets:
    records = list(dataset.records(record_set=rs.id))
    df = pd.DataFrame(records)
    dataframes[rs.id] = df
    print(f"Loaded {len(df)} records for record set {rs.id}")

# Show columns of the main record set
print(f"\nAvailable columns in the main record set:")
print(dataframes[main_record_set_id].columns.tolist())
dataframes[main_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and grouping data.

Select a numeric field and a grouping field from the available columns.

In [ ]:
# Select a numeric field and a group field by `@id`
# Example - assume 'cr:peptide_count' is a numeric field and 'cr:organism' is a group field
# Let's do a smart field selection:

# Try guessing column names for typical proteomics dataset
df = dataframes[main_record_set_id]
possible_numeric_fields = [col for col in df.columns if 'peptide' in col.lower() or 'abundance' in col.lower() or 'coverage' in col.lower() or df[col].dtype.kind in 'fi']
print(f"Possible numeric fields: {possible_numeric_fields}")
# Let's select the first one found
numeric_field = possible_numeric_fields[0]

# For grouping, look for any field with 'sample' or 'condition' or 'organism' in the name
possible_group_fields = [col for col in df.columns if 'sample' in col.lower() or 'condition' in col.lower() or 'organism' in col.lower()]
group_field = possible_group_fields[0] if possible_group_fields else None
print(f"Selected numeric field for analysis: {numeric_field}")
if group_field:
    print(f"Selected group field: {group_field}")

# Filtering for values > threshold (e.g. peptide count > 10)
threshold = 10
filtered_df = df[df[numeric_field].astype(float) > threshold]
print(f"Filtered records with '{numeric_field}' > {threshold}: {len(filtered_df)} records")
display(filtered_df.head())

# Normalization
filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field].astype(float) - filtered_df[numeric_field].astype(float).mean()) / filtered_df[numeric_field].astype(float).std()
print(f"Normalized {numeric_field} for filtered records:")
display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

# Group by group_field if available
if group_field and group_field in filtered_df.columns:
    grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
    print(f"Grouped data by {group_field} (mean {numeric_field}):")
    display(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships between fields.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of the numeric field
plt.figure(figsize=(7, 4))
sns.histplot(df[numeric_field].astype(float).dropna(), bins=30, kde=True, color='skyblue')
plt.title(f"Distribution of {numeric_field}")
plt.xlabel(numeric_field)
plt.ylabel("Count")
plt.show()

# Boxplot by group if group_field present
if group_field and group_field in df.columns:
    plt.figure(figsize=(10, 5))
    sns.boxplot(x=group_field, y=numeric_field, data=df)
    plt.title(f"{numeric_field} by {group_field}")
    plt.xticks(rotation=45)
    plt.show()

## 6. Conclusion
In this notebook, we explored the FAIR^2 dataset on mass spectrometry analysis of extracellular vesicles from human mast cells using the `mlcroissant` library. We loaded the dataset using its Croissant schema, reviewed the metadata and data structure, extracted and explored main records, performed basic filtering and normalization, grouped data, and visualized distributions. For further biological or clinical insights, refer to the data documentation and corresponding field descriptions accessible via each entity's `@id`.

This notebook can serve as a template for exploring Croissant-compliant datasets.
